# Install libraries

In [1]:
!pip install scikit-learn pandas numpy joblib

# Import libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

# Load Dataset

In [4]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

# View Dataset

In [5]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
print(movies.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB
None


In [8]:
print(ratings.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


# Check Missing Values

In [9]:
print(movies.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64


In [10]:
print(ratings.isnull().sum())

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


# Create Text Feature

In [11]:
movies["genres"] = movies["genres"].fillna("")

# TF-IDF Vectorization

In [12]:
tfidf = TfidfVectorizer(stop_words="english")

In [13]:
tfidf_matrix = tfidf.fit_transform(movies["genres"])

# Cosine Similarity

In [14]:
similarity = cosine_similarity(tfidf_matrix)

# Recommendation Function

In [15]:
def recommend(movie_name, top_n=10):

    movie_name = movie_name.lower()

    matches = movies[movies["title"].str.lower() == movie_name]

    if matches.empty:
        return []

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = []

    for i, score in similarity_scores:
        recommendations.append(movies.iloc[i]["title"])

    return recommendations

# Test

In [16]:
recommend("Toy Story (1995)")

['Antz (1998)',
 'Toy Story 2 (1999)',
 'Adventures of Rocky and Bullwinkle, The (2000)',
 "Emperor's New Groove, The (2000)",
 'Monsters, Inc. (2001)',
 'Wild, The (2006)',
 'Shrek the Third (2007)',
 'Tale of Despereaux, The (2008)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Turbo (2013)']

# Save Model Files

In [17]:
joblib.dump(tfidf, "tfidf.pkl")

['tfidf.pkl']

In [18]:
joblib.dump(similarity, "similarity.pkl")

['similarity.pkl']

In [19]:
joblib.dump(movies, "movies.pkl")

['movies.pkl']